# CityGuard — dziury i pęknięcia (YOLO12)

Fine-tuning `yolo12n.pt` na **całym RDD2022** (CRDDC / SEKILab): zdjęcia z kamer na pojazdach, 6 krajów, klasy D00/D10/D20/D40. Dron z Chin (`China_Drone`) pomijamy — inny kąt kadru.

Źródło: [sekilab/RoadDamageDetector](https://github.com/sekilab/RoadDamageDetector), [Figshare](https://figshare.com/articles/dataset/RDD2022_-_The_multi-national_Road_Damage_Dataset_released_through_CRDDC_2022/21431547). ~47 tys. zdjęć, 55 tys.+ boxów. Etykiety VOC XML → YOLO. Klasy scalamy do `crack` + `pothole`.

Śmieci trenuj osobno w `ml/train_litter.ipynb` (drugie konto Colab).

Środowisko: **GPU T4**. S3 SEKILab jest martwy. Notebook: Figshare (~13 GB, często **ZIP w ZIP-ie** + XML osobno od JPG) → Hugging Face `dronefreak/RDD2022` (już YOLO) → Kaggle. Na krótką sesję ustaw `MAX_IMAGES` albo wykreśl `Norway` z `COUNTRIES`.

In [ ]:
!pip -q install -U ultralytics kagglehub huggingface_hub

In [ ]:
import torch

!nvidia-smi
if not torch.cuda.is_available():
    raise SystemExit("Włącz GPU: Środowisko wykonawcze → Zmień typ środowiska → T4")
print("GPU:", torch.cuda.get_device_name(0))

## Konfiguracja

- `COUNTRIES` — które foldery wziąć z rozpakowanego RDD. `China_Drone` celowo nie ma na liście (inny kąt).
- `MAX_IMAGES = 0` — cały zbiór po konwersji. Na sesję 90 min ustaw `8000`.
- `SAVE_TO_DRIVE` — kopia `best.pt` po każdej epoce na Dysk Google.
- Opcjonalnie: `kaggle.json` w `/content` jako zapas, gdy Figshare nie pójdzie.

In [ ]:
COUNTRIES = [
    "Japan",
    "Czech",
    "India",
    "Norway",
    "United_States",
    "China_MotorBike",
]
MAX_IMAGES = 0  # 0 = wszystko; na T4 w 90 min spróbuj 8000
EPOCHS = 40
IMGSZ = 640
BATCH = 16
CONF = 0.35
SAVE_TO_DRIVE = True
KAGGLE_SLUG = "aliabdelmenam/rdd-2022"
FIGSHARE_URLS = [
    "https://ndownloader.figshare.com/files/38030910",
    "https://figshare.com/ndownloader/files/38030910",
]

In [ ]:
import json
import random
import shutil
import subprocess
import xml.etree.ElementTree as ET
from pathlib import Path

import yaml
from PIL import Image
from ultralytics import YOLO

IMAGE_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
RAW = Path("/content/raw/rdd2022")
DEST = Path("/content/datasets/road")

ROAD_MAP = {
    "D00": "crack",
    "D01": "crack",
    "D10": "crack",
    "D11": "crack",
    "D20": "crack",
    "D40": "pothole",
}
CLASS_NAMES = ["crack", "pothole"]


def run(cmd: list[str]) -> None:
    print("+", " ".join(cmd))
    subprocess.check_call(cmd)


def iter_xml(root: Path):
    for path in root.rglob("*"):
        if path.is_file() and path.suffix.lower() == ".xml":
            yield path


def has_xml(root: Path) -> bool:
    return next(iter_xml(root), None) is not None


def summarize(root: Path) -> None:
    xmls = list(iter_xml(root))
    images = [p for p in root.rglob("*") if p.suffix.lower() in IMAGE_EXT]
    zips = list(root.rglob("*.zip"))
    print("katalog:", root)
    print("xml:", len(xmls), "obrazy:", len(images), "zip:", len(zips))
    print("top:", [p.name for p in root.iterdir()][:25])
    if xmls:
        print("przykład xml:", xmls[0])
    if images:
        print("przykład obraz:", images[0])
    if zips:
        print("zagnieżdżone zip:", [p.name for p in zips[:8]])


def unpack_nested_zips(root: Path, skip_names: set[str]) -> None:
    seen: set[Path] = set()
    changed = True
    while changed:
        changed = False
        for zip_path in list(root.rglob("*.zip")):
            resolved = zip_path.resolve()
            if zip_path.name in skip_names or resolved in seen:
                continue
            seen.add(resolved)
            dest = zip_path.parent / zip_path.stem
            dest.mkdir(parents=True, exist_ok=True)
            print("rozpakowuję zagnieżdżony ZIP:", zip_path.name)
            try:
                run(["unzip", "-q", "-o", str(zip_path), "-d", str(dest)])
            except subprocess.CalledProcessError:
                fixed = zip_path.with_name(zip_path.stem + "_fixed.zip")
                run(["zip", "-FF", str(zip_path), "--out", str(fixed)])
                run(["unzip", "-q", "-o", str(fixed), "-d", str(dest)])
            changed = True


def ensure_kaggle_json() -> bool:
    dest = Path.home() / ".kaggle" / "kaggle.json"
    src = Path("/content/kaggle.json")
    if dest.exists():
        dest.chmod(0o600)
        return True
    if src.exists():
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(src, dest)
        dest.chmod(0o600)
        return True
    return False


def download_from_kaggle() -> Path | None:
    if not ensure_kaggle_json():
        print("Brak /content/kaggle.json — pomijam Kaggle")
        return None
    try:
        import kagglehub
    except ImportError:
        run(["pip", "install", "-q", "kagglehub"])
        import kagglehub
    print("Pobieram RDD2022 z Kaggle:", KAGGLE_SLUG)
    root = Path(kagglehub.dataset_download(KAGGLE_SLUG))
    if has_xml(root):
        return root
    print("Kaggle: brak XML w", root)
    return None


def download_from_figshare() -> Path:
    RAW.mkdir(parents=True, exist_ok=True)
    zip_path = RAW / "RDD2022_released_through_CRDDC2022.zip"
    if zip_path.exists() and zip_path.stat().st_size < 1_000_000_000:
        print("usuwam niepełny ZIP Figshare")
        zip_path.unlink()
    if not zip_path.exists() or zip_path.stat().st_size < 1_000_000_000:
        ok = False
        for url in FIGSHARE_URLS:
            try:
                run([
                    "wget", "-c", "--tries=5",
                    "--user-agent=Mozilla/5.0",
                    "-O", str(zip_path), url,
                ])
                if zip_path.exists() and zip_path.stat().st_size >= 1_000_000_000:
                    ok = True
                    break
            except subprocess.CalledProcessError as exc:
                print("Figshare wget padł:", url, exc)
        if not ok:
            raise SystemExit(
                "RDD2022: S3 SEKILab i Figshare nie działają. "
                "Wgraj kaggle.json do /content (dataset aliabdelmenam/rdd-2022) "
                "i odpal komórkę ponownie."
            )
    marker = RAW / ".extracted"
    if not (marker.exists() and has_xml(RAW)):
        try:
            run(["unzip", "-q", "-o", str(zip_path), "-d", str(RAW)])
        except subprocess.CalledProcessError:
            fixed = RAW / "RDD2022_fixed.zip"
            print("ZIP uszkodzony, próbuję zip --fixfix")
            run(["zip", "-FF", str(zip_path), "--out", str(fixed)])
            run(["unzip", "-q", "-o", str(fixed), "-d", str(RAW)])
        unpack_nested_zips(RAW, {zip_path.name, "RDD2022_fixed.zip"})
    summarize(RAW)
    if not has_xml(RAW):
        print("Figshare: nadal brak XML — spróbuję Hugging Face")
        return RAW
    marker.write_text("figshare", encoding="utf-8")
    return RAW


def download_from_hf() -> Path | None:
    try:
        from huggingface_hub import snapshot_download
    except ImportError:
        run(["pip", "install", "-q", "huggingface_hub"])
        from huggingface_hub import snapshot_download
    print("Pobieram RDD2022 (YOLO) z Hugging Face: dronefreak/RDD2022")
    root = Path(snapshot_download("dronefreak/RDD2022", repo_type="dataset"))
    data = root / "data"
    if (data / "labels").exists() and next((data / "labels").rglob("*.txt"), None):
        return data
    print("Hugging Face: brak etykiet YOLO w", root)
    return None


def download_rdd() -> Path:
    RAW.mkdir(parents=True, exist_ok=True)
    leftover = RAW / "RDD2022_Japan.zip"
    if leftover.exists() and leftover.stat().st_size < 50_000_000:
        leftover.unlink()
    if not has_xml(RAW):
        unpack_nested_zips(RAW, {"RDD2022_released_through_CRDDC2022.zip", "RDD2022_fixed.zip"})
    if has_xml(RAW):
        print("już rozpakowane:", RAW)
        summarize(RAW)
        return RAW
    figshare_root = download_from_figshare()
    if has_xml(figshare_root):
        return figshare_root
    hf_root = download_from_hf()
    if hf_root is not None:
        return hf_root
    kaggle_root = download_from_kaggle()
    if kaggle_root is not None:
        return kaggle_root
    summarize(RAW)
    raise FileNotFoundError(
        "RDD2022: brak XML/YOLO. Zostaw ZIP Figshare i odpal komórkę ponownie, "
        "albo wgraj kaggle.json do /content."
    )


def country_ok(path: Path) -> bool:
    hay = "/".join(path.parts).lower()
    if "/test/" in f"/{hay}/" or "china_drone" in hay:
        return False
    return any(name.lower() in hay for name in COUNTRIES)


def find_image(xml_path: Path) -> Path | None:
    for ext in IMAGE_EXT:
        cand = xml_path.with_suffix(ext)
        if cand.exists():
            return cand
    for parent in xml_path.parents:
        images_dir = parent / "images"
        if not images_dir.is_dir():
            continue
        for ext in IMAGE_EXT:
            cand = images_dir / f"{xml_path.stem}{ext}"
            if cand.exists():
                return cand
    return None


def remap_yolo_label(text: str) -> str:
    lines = []
    for line in text.splitlines():
        if not line.strip():
            continue
        parts = line.split()
        cls = int(float(parts[0]))
        if cls in {0, 1, 2}:
            parts[0] = "0"
        elif cls == 3:
            parts[0] = "1"
        else:
            continue
        lines.append(" ".join(parts))
    return "\n".join(lines)


def voc_to_yolo(xml_path: Path, image: Path) -> str:
    root = ET.parse(xml_path).getroot()
    size = root.find("size")
    width = float(size.findtext("width") or 0) if size is not None else 0
    height = float(size.findtext("height") or 0) if size is not None else 0
    if width <= 1 or height <= 1:
        with Image.open(image) as handle:
            width, height = handle.size
    lines = []
    for obj in root.findall("object"):
        raw = (obj.findtext("name") or "").strip()
        mapped = ROAD_MAP.get(raw)
        if mapped is None:
            continue
        box = obj.find("bndbox")
        if box is None:
            continue
        xmin = max(0.0, float(box.findtext("xmin") or 0))
        ymin = max(0.0, float(box.findtext("ymin") or 0))
        xmax = min(width, float(box.findtext("xmax") or 0))
        ymax = min(height, float(box.findtext("ymax") or 0))
        if xmax <= xmin or ymax <= ymin:
            continue
        cls = CLASS_NAMES.index(mapped)
        lines.append(
            f"{cls} {(xmin + xmax) / 2 / width:.6f} {(ymin + ymax) / 2 / height:.6f} "
            f"{(xmax - xmin) / width:.6f} {(ymax - ymin) / height:.6f}"
        )
    return "\n".join(lines)


def bump_country(path: Path, counts: dict[str, int]) -> None:
    hay = "/".join(path.parts).lower() + "/" + path.name.lower()
    for name in COUNTRIES:
        if name.lower() in hay:
            counts[name] += 1
            return


def collect_pairs() -> list[tuple[Path, str]]:
    labeled: list[tuple[Path, str]] = []
    counts: dict[str, int] = {name: 0 for name in COUNTRIES}
    root = download_rdd()
    for xml_path in iter_xml(root):
        if not country_ok(xml_path):
            continue
        image = find_image(xml_path)
        if image is None:
            continue
        text = voc_to_yolo(xml_path, image)
        if not text:
            continue
        labeled.append((image, text + "\n"))
        bump_country(xml_path, counts)
    if not labeled:
        labels_root = root / "labels" if (root / "labels").exists() else root
        for label_path in labels_root.rglob("*.txt"):
            if label_path.name == "data.yaml":
                continue
            if not country_ok(label_path) and not any(
                name.lower() in label_path.name.lower() for name in COUNTRIES
            ):
                continue
            image = None
            rel = label_path.relative_to(labels_root) if labels_root in label_path.parents or labels_root == label_path.parent else None
            images_root = root / "images"
            if rel is not None and images_root.exists():
                for ext in IMAGE_EXT:
                    cand = images_root / rel.with_suffix(ext)
                    if cand.exists():
                        image = cand
                        break
            if image is None:
                image = find_image(label_path)
            if image is None:
                continue
            text = remap_yolo_label(label_path.read_text(encoding="utf-8"))
            if not text:
                continue
            labeled.append((image, text + "\n"))
            bump_country(label_path, counts)
    for name, count in counts.items():
        print(f"{name}: +{count} obrazów z boxem")
    if not labeled:
        raise FileNotFoundError("RDD2022: zero par obraz+XML po filtrze klas D00/D10/D20/D40")
    random.Random(42).shuffle(labeled)
    if MAX_IMAGES and len(labeled) > MAX_IMAGES:
        labeled = labeled[:MAX_IMAGES]
        print("obcinam do MAX_IMAGES =", MAX_IMAGES)
    print("razem", len(labeled))
    return labeled


def link_split(pairs: list[tuple[Path, str]], dest: Path, names: list[str]) -> Path:
    if dest.exists():
        shutil.rmtree(dest)
    for split in ("train", "val"):
        (dest / "images" / split).mkdir(parents=True)
        (dest / "labels" / split).mkdir(parents=True)
    val_count = max(1, int(len(pairs) * 0.1))
    if len(pairs) < 2:
        raise RuntimeError(f"Za mało obrazów: {len(pairs)}")
    for index, (image, label) in enumerate(pairs):
        split = "val" if index < val_count else "train"
        stem = f"{index:05d}_{image.stem}"
        target = dest / "images" / split / f"{stem}{image.suffix.lower()}"
        target.symlink_to(image.resolve())
        (dest / "labels" / split / f"{stem}.txt").write_text(label, encoding="utf-8")
    yaml_path = dest / "cityguard.yaml"
    yaml_path.write_text(
        yaml.safe_dump(
            {
                "train": str((dest / "images" / "train").resolve()),
                "val": str((dest / "images" / "val").resolve()),
                "names": {i: name for i, name in enumerate(names)},
            },
            sort_keys=False,
            allow_unicode=True,
        ),
        encoding="utf-8",
    )
    print(f"road: {len(pairs) - val_count} train / {val_count} val, klasy {names}")
    return yaml_path


def backup_epoch(weight_name: str):
    def _hook(trainer):
        if not DRIVE_MOUNTED:
            return
        best = Path(trainer.save_dir) / "weights" / "best.pt"
        if not best.exists():
            return
        target = Path("/content/drive/MyDrive/cityguard")
        target.mkdir(parents=True, exist_ok=True)
        shutil.copy(best, target / weight_name)
        print(f"Drive: {weight_name} po epoce {trainer.epoch + 1}")

    return _hook


def fit_yolo(data_yaml: Path) -> Path:
    import torch

    def fit(current_batch: int):
        model = YOLO("yolo12n.pt")
        model.add_callback("on_fit_epoch_end", backup_epoch("best_road.pt"))
        model.train(
            data=str(data_yaml),
            epochs=EPOCHS,
            imgsz=IMGSZ,
            batch=current_batch,
            device=0,
            project="/content/runs",
            name="road",
            exist_ok=True,
            patience=10,
            workers=2,
            plots=True,
            save_period=5,
            cos_lr=True,
        )
        return model

    try:
        model = fit(BATCH)
    except RuntimeError as exc:
        if "out of memory" not in str(exc).lower():
            raise
        print("CUDA OOM. Powtarzam z batch=8.")
        torch.cuda.empty_cache()
        model = fit(8)

    best = Path("/content/runs/road/weights/best.pt")
    out = Path("/content/best_road.pt")
    shutil.copy(best, out)
    Path("/content/classes_road.json").write_text(
        json.dumps(
            {"task": "road", "conf": CONF, "names": {str(k): v for k, v in model.names.items()}},
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )
    print("Wagi:", out)
    return out


DRIVE_MOUNTED = False
if SAVE_TO_DRIVE:
    try:
        from google.colab import drive

        drive.mount("/content/drive")
        DRIVE_MOUNTED = True
    except Exception as exc:
        print("Drive nie wszedł — trenuję bez kopii na Dysku. Wagi będą w /content.")
        print(exc)

fit_yolo(link_split(collect_pairs(), DEST, CLASS_NAMES))

## Pobranie wag

Rozpakuj ZIP do `ml/weights/`. Inferencja szuka `best_road.pt`.

In [ ]:
import zipfile
from pathlib import Path

from google.colab import files

bundle = Path("/content/cityguard_best_road.pt.zip")
with zipfile.ZipFile(bundle, "w") as archive:
    found = False
    for name in ("best_road.pt", "classes_road.json"):
        path = Path("/content") / name
        if path.exists():
            archive.write(path, name)
            found = True
            print("pakuję", name)
        else:
            print("pomijam", name)
if not found:
    raise SystemExit("Nie ma jeszcze wag. Najpierw odpal trening.")
files.download(str(bundle))

## Podgląd na klatkach z demo

Ustaw `RUN_PREVIEW = True` i uruchom komórkę ręcznie. Próg: `CONF` (0.35).

In [ ]:
RUN_PREVIEW = False

if not RUN_PREVIEW:
    print("Podgląd wyłączony. Ustaw RUN_PREVIEW = True i odpal tę komórkę ponownie.")
else:
    from pathlib import Path

    from google.colab import files
    from ultralytics import YOLO

    preview = Path("/content/preview_images")
    preview.mkdir(exist_ok=True)
    for name, blob in files.upload().items():
        (preview / name).write_bytes(blob)
    YOLO("/content/best_road.pt").predict(
        source=str(preview),
        conf=CONF,
        save=True,
        project="/content/preview_out",
        name="road",
        exist_ok=True,
    )
    print("Obrazki z boxami: /content/preview_out/road")